# Lecture 7 — Class Exercise
## Part-to-Whole: Hierarchical Visualization

> **Push to:** `week07/lecture07_exercise.ipynb`


In [2]:
import pandas as pd
import plotly.express as px
import numpy as np

# Dataset: Global Energy Mix by Country and Source
df = pd.read_csv('global_energy_mix.csv')

# Source type mapping — reuse from lecture
source_category = {
    'Coal': 'Fossil', 'Oil': 'Fossil', 'Natural Gas': 'Fossil',
    'Nuclear': 'Low-carbon', 'Hydro': 'Low-carbon',
    'Wind': 'Renewable', 'Solar': 'Renewable', 'Other Renewables': 'Renewable'
}
df['Source_Type'] = df['Source'].map(source_category)

print(f"Loaded: {len(df)} rows")
print(df.head(10))

Loaded: 103 rows
         Country         Region            Source  Share_pct     TWh  \
0  United States  North America              Coal         10  1015.0   
1  United States  North America               Oil         35  3220.0   
2  United States  North America       Natural Gas         34  3083.0   
3  United States  North America           Nuclear          9   798.0   
4  United States  North America             Hydro          3   339.0   
5  United States  North America              Wind          4   413.0   
6  United States  North America             Solar          3   325.0   
7  United States  North America  Other Renewables          2   229.0   
8          China           Asia              Coal         58  6929.1   
9          China           Asia               Oil         18  1620.0   

  Source_Type  
0      Fossil  
1      Fossil  
2      Fossil  
3  Low-carbon  
4  Low-carbon  
5   Renewable  
6   Renewable  
7   Renewable  
8      Fossil  
9      Fossil  


## Task 1 — Treemap: fossil fuel dependency by country

**What to build:** A treemap showing **fossil fuel TWh only**, broken down by Region → Country → Source (Coal / Oil / Natural Gas).

**Requirements:**
- Filter to fossil sources only before plotting
- Use `path=['Region', 'Country', 'Source']` for the hierarchy
- Colour encodes the fossil source type (Coal / Oil / Natural Gas) with a CVD-safe palette
- Show TWh values in labels — no percentages
- Grey out parent nodes (Region and Country level)
- Insight title naming which region or country is most fossil-dependent

> 💡 `df.loc[df['Source_Type'] == 'Fossil']`


In [7]:
# Task 1
# YOUR CODE HERE

df_fossil = df[df["Source_Type"] == "Fossil"]

fig1 = px.treemap(
    df_fossil,
    path=["Region", "Country", "Source"],
    values="TWh",
    color="Source",
    color_discrete_map={
        "Coal": "#0072B2",
        "Oil": "#E69F00",
        "Natural Gas": "#009E73"
    },
    title="Fossil Fuel Energy by Region, Country and Source"
)

fig1.update_traces(root_color="lightgrey")
fig1.show()



## Task 2 — Treemap vs bar: low-carbon energy by country

**What to build:** Build **both** a treemap and a horizontal bar chart showing total low-carbon TWh (Nuclear + Hydro) per country. Then answer the question in a markdown cell below.

**Requirements:**
- Filter to `Source_Type == 'Low-carbon'` and aggregate TWh by country
- Treemap: single-level `path=['All', 'Country']` with a dummy root node labelled `'Low-carbon'`
- Bar chart: sorted by TWh, horizontal orientation, CVD-safe colour
- Both charts show TWh values, not percentages
- Insight title on the bar chart naming the leading country

In [8]:
# Task 2 — charts
# YOUR CODE HERE


low_carbon = df[df["Source_Type"] == "Low-carbon"]

country_energy = low_carbon.groupby("Country")["TWh"].sum().reset_index()


country_energy["All"] = "Low-carbon"

fig2 = px.treemap(
    country_energy,
    path=["All", "Country"],
    values="TWh",
    color="TWh",
    color_continuous_scale="Blues",
    title="Low-carbon Energy Production by Country"
)

fig2.show()


country_energy = country_energy.sort_values("TWh")

fig3 = px.bar(
    country_energy,
    x="TWh",
    y="Country",
    orientation="h",
    title="Countries Producing the Most Low-carbon Energy",
    color_discrete_sequence=["#0072B2"],
    text="TWh"
)

fig3.update_layout(showlegend=False)
fig3.show()


### Task 2 — reflection

**Which chart tells the ranking story more clearly, and why?**

*YOUR ANSWER HERE*
The bar chart shows the ranking more clearly because it is easier to compare the energy production of each country.

## Task 3 — 100% stacked bar: which region relies most on coal?

**What to build:** A horizontal 100% stacked bar chart showing, for each **Region**, how its fossil energy splits between Coal, Oil and Natural Gas. This is the **same data as your Task 1 treemap**, visualised a second way.

**Requirements:**
- Reuse `df_fossil` from Task 1; aggregate TWh by `Region` × `Source` and convert to **shares of each region's fossil total**
- Horizontal bars, regions **sorted by coal share** (highest at the top)
- Coal is the segment on the baseline (first in `category_orders`) and the **only coloured segment** — keep it the same colour as in Task 1; Oil and Natural Gas in two greys
- Percentage labels on the coal segments only
- Insight title naming the most coal-dependent region and its coal share

> 💡 `df.groupby('Region')['TWh'].transform('sum')` gives each row its region total

In [6]:
# Task 3
# YOUR CODE HERE
region_energy = df_fossil.groupby(
    ["Region", "Source"]
)["TWh"].sum().reset_index()
region_energy["Total"] = region_energy.groupby("Region")["TWh"].transform("sum")
region_energy["Share"] = region_energy["TWh"] / region_energy["Total"] * 100

coal = region_energy[region_energy["Source"] == "Coal"]
coal = coal.sort_values("Share", ascending=False)

regions = coal["Region"].tolist()


fig4 = px.bar(
    region_energy,
    x="Share",
    y="Region",
    color="Source",
    orientation="h",
    barmode="stack",
    category_orders={
        "Region": regions,
        "Source": ["Coal", "Oil", "Natural Gas"]
    },
    color_discrete_map={
        "Coal": "#0072B2",
        "Oil": "#BDBDBD",
        "Natural Gas": "#777777"
    },
    title="Coal Share in Regional Fossil Fuel Energy"
)


fig4.update_traces(
    texttemplate="%{x:.1f}%",
    textposition="inside",
    selector=dict(name="Coal")
)

fig4.update_layout(
    xaxis_title="Share of fossil energy (%)",
    yaxis_title="Region"
)

fig4.show()


### Task 3 — reflection

**Your Task 1 treemap and this stacked bar show the same fossil data. Name one question each chart answers that the other cannot.**

*YOUR ANSWER HERE*

1 — Treemap: Which country has the highest fossil energy consumption?

Ans: The treemap helps identify the country with the highest fossil energy consumption and compare different energy sources.

2 — Stacked Bar Chart: Which region has the highest percentage of coal compared with oil and natural gas?

Ans: The stacked bar chart helps compare the percentage share of coal, oil, and natural gas in each region.